In [ ]:
import sys
import math
from enum import Enum

print("=== Heap Diagnostic Code Loaded ===")

class Policy(Enum):
    FIRST_FIT = "first_fit"
    BEST_FIT = "best_fit"
    POOLING = "pooling"

class Heap:
    def __init__(self, size, policy=Policy.FIRST_FIT):
        self.blocks = [(0, size, True)]  # (start, size, free)
        self.policy = policy

        if policy == Policy.BEST_FIT:
            self.malloc_func = self.malloc_best_fit
        elif policy == Policy.FIRST_FIT:
            self.malloc_func = self.malloc_first_fit
        elif policy == Policy.POOLING:
            self.malloc_func = self.malloc_pooling
            self.prealloc_pooling(size)
        else:
            raise ValueError("Invalid policy")

    def prealloc_pooling(self, size):
        block_size = 16
        self.blocks.clear()
        if size < block_size:
            raise ValueError("Size must be at least 16")
        blocks = size // block_size;

        start = 0
        for i in range(0, blocks):
            self.blocks.append((start, 16, True))
            start += block_size

    def dump(self):
        print("Heap:", self.blocks)

    def malloc(self, size):
        print(f"\nCALL: malloc({size})")

        start = self.malloc_func(size)
        if start != -1:
            return start

        # ---- FORCED FAILURE PATH ----
        free_blocks = [bsize for _, bsize, free in self.blocks if free]
        total_free = sum(free_blocks)
        largest = max(free_blocks) if free_blocks else 0

        print(">>> ALLOCATION FAILED <<<")
        print("TOTAL FREE :", total_free)
        print("LARGEST BLK:", largest)
        print("REQUESTED :", size)

        if total_free >= size:
            raise RuntimeError("FRAGMENTATION ERROR DETECTED")
        else:
            raise RuntimeError("OUT OF MEMORY")

    def malloc_best_fit(self, size):
        index = -1
        diff = sys.maxsize
        for i, (start, bsize, free) in enumerate(self.blocks):
            if free and bsize >= size:
                temp_diff = bsize - size
                if(temp_diff < diff):
                    diff = temp_diff
                    index = i
        if index != -1:
            block = self.blocks[index]
            start, bsize, _ = block
            self.blocks[index] = (start, size, False)
            if bsize > size:
                self.blocks.insert(index + 1, (start + size, bsize - size, True))
            print("ALLOCATED")
            return start
        return -1

    def malloc_first_fit(self, size):
        for i, (start, bsize, free) in enumerate(self.blocks):
            if free and bsize >= size:
                self.blocks[i] = (start, size, False)
                if bsize > size:
                    self.blocks.insert(i + 1, (start + size, bsize - size, True))
                print("ALLOCATED")
                return start
        return -1
    
    def malloc_pooling(self, size):
        block_size = 16

        needed = (size + block_size - 1) // block_size

        count = 0
        start_index = -1

        for i, (start, bsize, free) in enumerate(self.blocks):
            if free:
                if count == 0:
                    start_index = i
                count += 1

                if count == needed:
                    for j in range(start_index, start_index + needed):
                        s, bs, _ = self.blocks[j]
                        self.blocks[j] = (s, bs, False)

                    print("ALLOCATED")
                    return self.blocks[start_index][0]
            else:
                count = 0
        return -1


    def free(self, ptr):
        print(f"\nCALL: free({ptr})")
        for i, (start, size, free) in enumerate(self.blocks):
            if start == ptr and not free:
                self.blocks[i] = (start, size, True)
                print("FREED")
                return
        raise RuntimeError("INVALID FREE")


=== Heap Diagnostic Code Loaded ===


In [10]:
heap = Heap(100)
heap.dump()

p1 = heap.malloc(20)
heap.dump()

p2 = heap.malloc(30)
heap.dump()

heap.free(p1)
heap.dump()

heap.free(p2)
heap.dump()

Heap: [(0, 100, True)]

CALL: malloc(20)
ALLOCATED
Heap: [(0, 20, False), (20, 80, True)]

CALL: malloc(30)
ALLOCATED
Heap: [(0, 20, False), (20, 30, False), (50, 50, True)]

CALL: free(0)
FREED
Heap: [(0, 20, True), (20, 30, False), (50, 50, True)]

CALL: free(20)
FREED
Heap: [(0, 20, True), (20, 30, True), (50, 50, True)]


In [12]:
heap = Heap(100, Policy.POOLING)

a = heap.malloc(20)   # 0–19
b = heap.malloc(20)   # 20–39
c = heap.malloc(20)   # 40–59
heap.dump()
d = heap.malloc(20)   # 60–79
heap.dump()

heap.free(b)          # free 20–39
heap.free(d)          # free 60–79
heap.dump()

# Total free = 40
# Largest block = 20
heap.malloc(25)       # MUST FAIL (fragmentation)


CALL: malloc(20)

CALL: malloc(20)

CALL: malloc(20)
Heap: [(0, 16, False), (16, 16, False), (32, 16, False), (48, 16, False), (64, 16, False), (80, 16, False)]

CALL: malloc(20)
>>> ALLOCATION FAILED <<<
TOTAL FREE : 0
LARGEST BLK: 0
REQUESTED : 20


RuntimeError: OUT OF MEMORY

In [ ]:
heap = Heap(50)

p1 = heap.malloc(30)
p2 = heap.malloc(15)

heap.dump()

# Truly out of memory
p3 = heap.malloc(10)

In [ ]:
heap.free(999)   # Invalid pointer